# `oil_network` — fresh-machine setup

Bootstraps a freshly-cloned copy of this repo into a working state:

1. Check Python + Postgres are reachable.
2. `pip install -r requirements.txt` into the active interpreter.
3. Capture your EIA API key + Postgres credentials into a `.env` file at the repo root.
4. Provision the Postgres role `eia_user` and database `eia_crude` (only if missing).
5. Run the master orchestrator end-to-end.

**Before running this notebook**, on a brand-new machine, do this in a terminal:

```bash
git clone <your-private-repo-url> oil-network
cd oil-network
python -m venv .venv
# Windows PowerShell:
.\.venv\Scripts\Activate.ps1
# macOS / Linux:
# source .venv/bin/activate
pip install jupyter python-dotenv
jupyter notebook setup.ipynb
```

Then open this notebook in the kernel pointing at `.venv` and run each cell top-to-bottom.

PostgreSQL needs to be installed and running locally on port 5432. On Windows the simplest path is the EDB installer (https://www.postgresql.org/download/windows/); on macOS, `brew install postgresql@16 && brew services start postgresql@16`. The notebook checks reachability before trying to write anything.

## 1. Prerequisite check — Python + Postgres reachable

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / 'code' / 'initialize_oil_network.ipynb').exists(), (
    f"Run this notebook from the repo root. Currently in: {REPO_ROOT}")

print(f'Python:      {sys.version.split()[0]}  ({sys.executable})')
print(f'Repo root:   {REPO_ROOT}')
assert sys.version_info >= (3, 11), 'Python 3.11+ required'
print('OK')

## 2. Install Python dependencies

Installs everything in `requirements.txt` into the active interpreter. Safe to re-run — pip skips packages already at the required version.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError(f'pip install failed (exit {result.returncode})')
print('OK')

## 3. Capture credentials into `.env`

Writes a `.env` file at the repo root. **This file is gitignored** — your key never leaves your machine.

Edit the values below before running this cell:
- `EIA_API_KEY` — get a free one at https://www.eia.gov/opendata/ (instant, just an email).
- The `PG_*` defaults match what the codebase expects; leave them alone unless your local Postgres uses different credentials.
- `PG_ADMIN_PASS` — your Postgres superuser (usually `postgres`) password. Only used in step 4 to provision the role + database. **Not written to `.env`.**

In [ ]:
# ── EDIT THESE ─────────────────────────────────────────────────────────
EIA_API_KEY    = ''                  # paste your key here

PG_HOST        = 'localhost'
PG_PORT        = '5432'
PG_DB          = 'eia_crude'
PG_USER        = 'eia_user'
PG_PASS        = 'eia_password'
PG_SCHEMA      = 'oil_network_data_loader'

# Used in step 4 only, not persisted.
PG_ADMIN_USER  = 'postgres'
PG_ADMIN_PASS  = ''                  # your local postgres superuser password
# ───────────────────────────────────────────────────────────────────────

assert EIA_API_KEY, 'Set EIA_API_KEY in this cell before running'

env_path = REPO_ROOT / '.env'
env_path.write_text(
    f'# Local environment — gitignored. Edit via setup.ipynb on each machine.\n'
    f'EIA_API_KEY={EIA_API_KEY}\n'
    f'PG_HOST={PG_HOST}\n'
    f'PG_PORT={PG_PORT}\n'
    f'PG_DB={PG_DB}\n'
    f'PG_USER={PG_USER}\n'
    f'PG_PASS={PG_PASS}\n'
    f'PG_SCHEMA={PG_SCHEMA}\n',
    encoding='utf-8',
)
print(f'Wrote {env_path} ({env_path.stat().st_size} bytes)')
print('(EIA_API_KEY redacted in display, full value written to file)')

# Also push into this kernel's environment so cell 4-5 see them.
import os
for k, v in [('EIA_API_KEY', EIA_API_KEY), ('PG_HOST', PG_HOST), ('PG_PORT', PG_PORT),
             ('PG_DB', PG_DB), ('PG_USER', PG_USER), ('PG_PASS', PG_PASS),
             ('PG_SCHEMA', PG_SCHEMA)]:
    os.environ[k] = v

## 4. Provision Postgres role + database

Connects as the superuser using `PG_ADMIN_PASS` from cell 3, then:

1. Creates role `eia_user` with password `eia_password` if missing.
2. Creates database `eia_crude` owned by `eia_user` if missing.
3. Grants `eia_user` CREATEDB so the orchestrator can drop+recreate schemas inside `eia_crude`.

Idempotent — re-running is a no-op once everything exists. If the role / DB already exist with different config, this cell does not touch them (you'd need to fix manually).

In [ ]:
import psycopg2
from psycopg2 import sql

# Try connecting as eia_user first — if that works the provisioning is already done.
try:
    with psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB,
                          user=PG_USER, password=PG_PASS) as conn:
        with conn.cursor() as cur:
            cur.execute('SELECT current_user, current_database()')
            u, d = cur.fetchone()
            print(f'OK — {u} can already reach {d}. Skipping provisioning.')
            already_provisioned = True
except psycopg2.OperationalError as e:
    already_provisioned = False
    print(f'Pre-check failed: {str(e).splitlines()[0]}')
    print('Will attempt provisioning as superuser...')

if not already_provisioned:
    assert PG_ADMIN_PASS, 'Set PG_ADMIN_PASS in cell 3 to provision the role+DB'
    # Connect to default postgres DB as admin to create the role + target DB.
    admin_conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname='postgres',
                                  user=PG_ADMIN_USER, password=PG_ADMIN_PASS)
    admin_conn.autocommit = True
    with admin_conn.cursor() as cur:
        cur.execute('SELECT 1 FROM pg_roles WHERE rolname = %s', (PG_USER,))
        if cur.fetchone():
            print(f'  role {PG_USER} already exists')
        else:
            cur.execute(sql.SQL('CREATE ROLE {} LOGIN PASSWORD {} CREATEDB').format(
                sql.Identifier(PG_USER), sql.Literal(PG_PASS)))
            print(f'  + created role {PG_USER}')
        cur.execute('SELECT 1 FROM pg_database WHERE datname = %s', (PG_DB,))
        if cur.fetchone():
            print(f'  database {PG_DB} already exists')
        else:
            cur.execute(sql.SQL('CREATE DATABASE {} OWNER {}').format(
                sql.Identifier(PG_DB), sql.Identifier(PG_USER)))
            print(f'  + created database {PG_DB} owned by {PG_USER}')
    admin_conn.close()
    with psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB,
                          user=PG_USER, password=PG_PASS) as conn:
        with conn.cursor() as cur:
            cur.execute('SELECT version()')
            print(f'OK — connected as {PG_USER} to {PG_DB}: {cur.fetchone()[0].splitlines()[0]}')

## 5. Run the master orchestrator

Spawns `code/initialize_oil_network.ipynb` as a subprocess. It runs all four stages:

1. **Asset graph** (~30 s) — `DROP SCHEMA oil_network CASCADE` → build schema → load asset_graph.json
2. **Metadata** (~5 s) — typed metadata for production nodes
3. **Source-data loaders** (~3–10 min) — fetches all 27 EIA crude datasets via the API
4. **Assignments** (~1 min) — formulas + migrations + views + resolver + audits

Expected end state: 252 nodes, 90 TS-bound vars, ~230k resolved values, 0 unresolved, 0 partition gaps, 0 capacity violations. See `claude/PROJECT_STATE.md` for the canonical numbers.

Total runtime: typically 3–10 minutes depending on EIA API latency.

In [ ]:
code_dir = REPO_ROOT / 'code'
orchestrator = code_dir / 'initialize_oil_network.ipynb'

proc = subprocess.Popen(
    [sys.executable, '-m', 'jupyter', 'nbconvert',
     '--to', 'notebook', '--execute', '--inplace',
     '--ExecutePreprocessor.timeout=1800',
     str(orchestrator)],
    cwd=str(code_dir), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nOrchestrator exit code: {proc.returncode}')
if proc.returncode != 0:
    raise RuntimeError('Orchestrator failed — open code/initialize_oil_network.ipynb in Jupyter to see per-stage output and stderr')

## 6. Sanity check — current state

Reads back the post-orchestrator state and prints the key invariants. Should match `claude/PROJECT_STATE.md` §3.1.

In [ ]:
with psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB,
                      user=PG_USER, password=PG_PASS) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT COUNT(*) FROM oil_network.assets')
        n_assets = cur.fetchone()[0]
        cur.execute('SELECT COUNT(*) FROM oil_network.variables')
        n_vars = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM oil_network.scenario_resolved_values WHERE source = 'unresolved'")
        n_unresolved = cur.fetchone()[0]
        cur.execute('SELECT MAX(run_id) FROM oil_network.scenario_resolver_runs')
        run_id = cur.fetchone()[0]

print(f'Assets:        {n_assets}')
print(f'Variables:     {n_vars}')
print(f'Unresolved:    {n_unresolved}     (must be 0)')
print(f'Last run_id:   {run_id}')

assert n_unresolved == 0, 'Resolver left unresolved rows — investigate'
print('\nAll invariants green. See `claude/PROJECT_STATE.md` §3 for the full picture.')

## 7. Next steps

Setup done. From here, the project state lives in:

- **`claude/CLAUDE.md`** — design principles + framework context (read first if returning after a break).
- **`claude/HANDOVER.md`** — pass-by-pass history; latest pass is the most relevant.
- **`claude/PROJECT_STATE.md`** — current numbers + outstanding work list in priority order.
- **`outputs/html/`** — 5 visualisation HTMLs (regenerate with `python code/regenerate_htmls.py --force`).

To start a substantive session, open Claude Code with this directory as the working directory — it'll pick up `CLAUDE.md` automatically.